In [2]:
# https://blog.csdn.net/mfsdmlove/article/details/126082421
import pandas as pd
import numpy as np
import scipy.optimize as sco

In [3]:
data = pd.read_excel(r'C:\code\python\optimization\sci_weight\收益率.xlsx',index_col=0)

In [3]:
data


,BMW,ORACLE,YOIGO,SACYR,BALAY,APPLE
Year,,,,,,
2011,1.4866,0.0562,0.9641,-1.2367,0.0512,0.5075
2012,0.2902,0.4356,0.2111,4.0332,0.2491,0.4014
2013,0.1911,0.8831,0.3746,0.3813,0.1815,-1.6639
2014,0.1288,0.3643,-0.2525,0.0666,0.2566,0.1564
2015,-0.0850,0.1460,0.4236,0.4114,0.4718,0.2199
2016,0.1068,0.0836,0.1333,-0.9241,0.3121,-0.0216
2017,-0.0040,-0.6284,-0.3053,3.4962,-0.0368,-0.0636
2018,0.0726,0.4273,-0.2470,0.4870,0.1204,-0.2773
2019,-0.2030,-0.2196,0.1404,0.0862,-0.3052,-0.2985


In [14]:
# 定义无风险收益率
rf = 0.03
# 定义资产数量
number_assets = 6

In [9]:
# 获取股票平均收益率
returns = data.mean() 

In [10]:
returns

BMW       0.19404
ORACLE    0.16162
YOIGO     0.13667
SACYR     0.56417
BALAY     0.11506
APPLE     0.20152
dtype: float64

In [12]:
# 获取股票收益率的方差协方差矩阵
cov = data.cov() 

In [13]:
cov

,BMW,ORACLE,YOIGO,SACYR,BALAY,APPLE
BMW,0.226425,0.020410,0.133854,-0.195604,0.007056,0.007589
ORACLE,0.020410,0.166990,0.031932,-0.107495,0.045059,-0.145486
YOIGO,0.133854,0.031932,0.152022,-0.224752,0.015483,-0.050985
SACYR,-0.195604,-0.107495,-0.224752,3.288723,0.040210,-0.509356
BALAY,0.007056,0.045059,0.015483,0.040210,0.053622,-0.074360
APPLE,0.007589,-0.145486,-0.050985,-0.509356,-0.074360,1.372986


In [15]:
# 给定权重，求组合收益率、标准差、夏普比率
def statistics(weights):        
    weights = np.array(weights)
    pret = np.dot(weights, returns) #获取组合收益率
    pvol = np.sqrt(np.dot(weights.T, np.dot(cov, weights))) #获取组合标准差
    psharpe = (pret - rf) / pvol #获取组合夏普比率
    return np.array([pret, pvol, psharpe])


In [16]:
# 设置目标：夏普比率最大
def minus_sharpe(x):
    return -statistics(x)[2]

# 约束条件分为eq和ineq
# eq表示函数结果等于0 ；ineq 表示 表达式大于等于0  
# 约束条件：权重之和=1
def constraint1(x):
    return np.sum(x) -1

con1 = {'type': 'eq', 'fun': constraint1} 

# 定义边界约束（优化变量的上下限）
bnds = tuple((None, 0.2) for x in range(number_assets)) #每个资产权重取值范围为0-0.2

# 设置初始权重
x0 = number_assets * [1 / number_assets]

# 规划求解
solution = sco.minimize(minus_sharpe,x0,method='SLSQP', bounds=bnds, constraints=con1)

# 求解出权重、组合收益率、标准差、夏普比率
final_weights = solution.x #权重
finanl_ratio = statistics(final_weights) #组合收益率、标准差、夏普比率

print(f'{final_weights[0]:.1%}投资于{data.columns[0]}，{final_weights[1]:.1%}投资于{data.columns[1]}，'
     f'{final_weights[2]:.1%}投资于{data.columns[2]}，{final_weights[3]:.1%}投资于{data.columns[3]}，'
     f'{final_weights[4]:.1%}投资于{data.columns[4]}，{final_weights[5]:.1%}投资于{data.columns[5]}，'
     f'组合收益率：{finanl_ratio[0]:.1%}，组合标准差：{finanl_ratio[1]:.1%},组合夏普比率：{finanl_ratio[2]:.1%}')


18.5%投资于BMW，20.0%投资于ORACLE，20.0%投资于YOIGO，9.7%投资于SACYR，20.0%投资于BALAY，11.7%投资于APPLE，组合收益率：19.7%，组合标准差：22.3%,组合夏普比率：75.0%


In [19]:
finanl_ratio

array([0.19723968, 0.22290366, 0.75027787])